In [0]:
from pyspark.sql.functions import broadcast

customers = spark.createDataFrame(
    [(1, "Alice"), (2, "Bob"), (3, "Carol")], ["cust_id", "name"]
)
orders = spark.createDataFrame(
    [(101, 1, 50.0), (102, 1, 20.0), (103, 2, 75.0), (104, 4, 30.0)],
    ["order_id", "cust_id", "amount"]
)
customers.write.mode("overwrite").saveAsTable("study_catalog.sec1.customers")
orders.write.mode("overwrite").saveAsTable("study_catalog.sec1.orders")

# Inner join: only rows with a match on both sides — order 104 (cust_id=4) drops out
inner_result = orders.join(customers, on="cust_id", how="inner")
inner_result.show()

# Left join: keep every order, even if the customer doesn't exist — order 104 gets a null name
left_result = orders.join(customers, on="cust_id", how="left")
left_result.show()

# Broadcast join: hint Spark to send the small `customers` table to every executor
# instead of shuffling both sides — same result as inner join, different execution plan
broadcast_result = orders.join(broadcast(customers), on="cust_id", how="inner")
broadcast_result.show()
broadcast_result.explain()  # look for "BroadcastHashJoin" in the physical plan output

# Union: stack two DataFrames with the same schema (row-wise concatenation)
more_orders = spark.createDataFrame([(105, 3, 15.0)], ["order_id", "cust_id", "amount"])
unioned = orders.union(more_orders)
print(orders.count(), more_orders.count(), unioned.count())  # 4, 1, 5 — union doesn't dedupe

In [0]:
%sql
select * from study_catalog.sec1.customers;
select * from study_catalog.sec1.orders;